
# PCA From Scratch → scikit-learn → Iris Classifier (Linear Regression OVR)

**Goal**: Go from first principles to practice.

1. **PCA from scratch** (centering, covariance, eigen-decomposition, projections, explained variance).
2. **Trivial 2D example** to build intuition.
3. **PCA with scikit-learn** (sanity-check our math).
4. **Classification on Iris** with a **linear regression One-vs-Rest (OVR)** classifier:
   - Pipelines: **StandardScaler**, optional **PCA**.
   - **k-fold cross validation** (stratified).
   - Metrics **before vs after PCA** on **train** and **test**:
     - $R^2$ (macro over one-vs-rest regressions)
     - Precision / Recall / F1 (macro, weighted)
     - ROC AUC (one-vs-rest, macro)
     - Confusion matrix
5. **Fisher's Linear Discriminant separability score**: For each principal component, compute
   $$ \text{Fisher} = \frac{(\mu_1 - \mu_2)^2}{\sigma_1^2 + \sigma_2^2} $$
   averaged over class pairs, and show that **components with larger eigenvalues tend to be more class-separable**.

All code cells are small and composable, with explanations and math along the way.



## 0. Environment & Imports

We avoid external internet access. Datasets come from `sklearn`. Plots use `matplotlib` only.


In [ ]:

import numpy as np
import itertools
import math
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as SKPCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_fscore_support,
                             roc_auc_score, r2_score)

np.random.seed(42)
plt.rcParams["figure.figsize"] = (6,4)
plt.rcParams["axes.grid"] = True

print("Environment ready.")



## 1. PCA: Intuition & Math

Given data matrix $X \in \mathbb{R}^{n \times d}$, with rows as samples and columns as features:

1. **Center** the data: $\tilde{X} = X - \mathbf{1}\mu^\top$, where $\mu$ is the column-wise mean.
2. **Covariance**: $C = \frac{1}{n-1}\tilde{X}^\top \tilde{X}$.
3. **Eigen-decomposition**: $C = V \Lambda V^\top$ with eigenvalues $\lambda_1 \ge \lambda_2 \ge \cdots \ge \lambda_d \ge 0$.
4. **Principal components**: columns of $V$; **explained variance** equals the eigenvalues; **explained variance ratio** is $\lambda_k / \sum_j \lambda_j$.
5. **Projection** onto first $m$ components: $Z = \tilde{X} V_{:,1:m}$.

PCA finds orthogonal directions of **maximal variance** — often capturing the “signal” using fewer dimensions.



### 1.1 PCA from Scratch Utilities


In [ ]:

def pca_from_scratch(X, n_components=None):
    # Returns (mean, components, eigenvalues, explained_variance_ratio, Z, C)
    X = np.asarray(X)
    mu = X.mean(axis=0)
    Xc = X - mu
    C = (Xc.T @ Xc) / (len(Xc) - 1)
    eigvals, eigvecs = np.linalg.eigh(C)  # ascending
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    if n_components is not None:
        eigvecs = eigvecs[:, :n_components]
        eigvals = eigvals[:n_components]
    total_var = np.linalg.eigvalsh(C).sum()
    evr = eigvals / total_var
    Z = (Xc @ eigvecs)
    return mu, eigvecs, eigvals, evr, Z, C

def explained_variance_ratio_from_cov(C):
    vals = np.linalg.eigvalsh(C)
    vals = np.sort(vals)[::-1]
    return vals / vals.sum(), vals



## 2. Trivial 2D Example

Create a small, correlated 2D dataset. We'll compute PCA **by hand** and visualize projections.


In [ ]:

n = 50
x = np.linspace(-2, 2, n)
y = 2.0 * x + 0.3 * np.random.randn(n)
X2 = np.c_[x, y]

mu2, comps2, vals2, evr2, Z2, C2 = pca_from_scratch(X2, n_components=2)

print("Mean:", mu2)
print("Eigenvalues (descending):", vals2)
print("Explained variance ratio:", evr2)

X2c = X2 - mu2
plt.scatter(X2c[:,0], X2c[:,1], alpha=0.7)
for i in range(2):
    vec = comps2[:, i] * math.sqrt(vals2[i]) * 2.5
    plt.plot([0, vec[0]], [0, vec[1]], linewidth=2, label=f"PC{i+1}")
plt.axhline(0, linestyle="--", linewidth=1)
plt.axvline(0, linestyle="--", linewidth=1)
plt.legend()
plt.title("Centered data with principal axes")
plt.xlabel("x (centered)")
plt.ylabel("y (centered)")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10,4))
axes[0].hist(Z2[:,0], bins=12)
axes[0].set_title("Projection on PC1")
axes[1].hist(Z2[:,1], bins=12)
axes[1].set_title("Projection on PC2")
plt.show()



## 3. Sanity Check with scikit-learn


In [ ]:

from sklearn.decomposition import PCA as SKPCA
sk = SKPCA(n_components=2).fit(X2)
print("sklearn explained variance:", sk.explained_variance_)
print("sklearn explained variance ratio:", sk.explained_variance_ratio_)
print("Scratch PC1 (unit):", comps2[:,0])
print("SK PC1 (unit):     ", sk.components_[0])
print("Cosine similarity:", np.dot(comps2[:,0], sk.components_[0]))



## 4. Fisher's Linear Discriminant Separability (1D)
For two classes with projected scalars $z^{(1)}, z^{(2)}$:
$$
S_F = \frac{(\mu_1 - \mu_2)^2}{\sigma_1^2 + \sigma_2^2}.
$$
For multiclass, we average over unordered class pairs.


In [ ]:

def fisher_pair_score(z1, z2):
    m1, m2 = z1.mean(), z2.mean()
    v1, v2 = z1.var(ddof=1), z2.var(ddof=1)
    denom = v1 + v2
    return 0.0 if denom == 0 else (m1 - m2)**2 / denom

def fisher_multiclass_1d(z, y):
    # Average Fisher score over all unordered class pairs for a 1D projection z.
    import itertools
    classes = np.unique(y)
    pairs = list(itertools.combinations(classes, 2))
    scores = []
    for a, b in pairs:
        scores.append(fisher_pair_score(z[y==a], z[y==b]))
    return np.mean(scores), dict(zip(pairs, scores))



## 5. Iris: PCA + Fisher vs Eigenvalues
We standardize, compute PCA (scratch), and measure Fisher separability for each PC.


In [ ]:

iris = load_iris()
X = iris.data.copy()
y = iris.target.copy()

scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)

mu, comps, vals, evr, Z, C = pca_from_scratch(Xs, n_components=4)

fisher_scores = []
for k in range(Z.shape[1]):
    s, _ = fisher_multiclass_1d(Z[:,k], y)
    fisher_scores.append(s)

print("Eigenvalues:", vals)
print("Explained variance ratio:", evr)
print("Fisher scores per PC:", fisher_scores)

fig, ax = plt.subplots()
ax.plot(range(1, len(vals)+1), vals, marker="o")
ax.set_xlabel("PC index")
ax.set_ylabel("Eigenvalue")
ax.set_title("PCA Eigenvalues (Iris, scaled)")
plt.show()

fig, ax = plt.subplots()
ax.plot(range(1, len(fisher_scores)+1), fisher_scores, marker="o")
ax.set_xlabel("PC index")
ax.set_ylabel("Fisher separability (avg over pairs)")
ax.set_title("Fisher Scores per PC")
plt.show()



## 6. OVR Linear Regression Classifier
Train one regressor per class to predict the class indicator (0/1).


In [ ]:

class OVRLinearRegression:
    def __init__(self):
        self.regressors_ = []
        self.classes_ = None

    def fit(self, X, y):
        X = np.asarray(X); y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.regressors_ = []
        for c in self.classes_:
            r = LinearRegression()
            r.fit(X, (y == c).astype(float))
            self.regressors_.append(r)
        return self

    def decision_function(self, X):
        X = np.asarray(X)
        scores = np.column_stack([r.predict(X) for r in self.regressors_])
        return scores

    def predict(self, X):
        scores = self.decision_function(X)
        return self.classes_[np.argmax(scores, axis=1)]

    def score_individual_r2(self, X, y):
        # Return list of R^2 per class (OVR) and macro average
        y = np.asarray(y)
        r2s = []
        for c, r in zip(self.classes_, self.regressors_):
            y_bin = (y == c).astype(float)
            r2s.append(r2_score(y_bin, r.predict(X)))
        return r2s, float(np.mean(r2s))



## 7. Train/Test: Baseline vs PCA(2)
We compare **Scaler → OVR LR** to **Scaler → PCA(2) → OVR LR** and report metrics for train and test.


In [ ]:

from sklearn.decomposition import PCA as SKPCA

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,
                                                    random_state=42, stratify=y)

def fit_pipeline(use_pca=False, n_components=2):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xte = scaler.transform(X_test)
    if use_pca:
        pca = SKPCA(n_components=n_components, random_state=42)
        Xtr = pca.fit_transform(Xtr)
        Xte = pca.transform(Xte)
    else:
        pca = None
    clf = OVRLinearRegression().fit(Xtr, y_train)
    return scaler, pca, clf, Xtr, Xte

def evaluate_pipeline(name, scaler, pca, clf, Xtr, Xte):
    # Train
    yhat_tr = clf.predict(Xtr)
    scores_tr = clf.decision_function(Xtr)
    r2s_tr, r2_macro_tr = clf.score_individual_r2(Xtr, y_train)
    pr_tr, rc_tr, f1_tr, _ = precision_recall_fscore_support(y_train, yhat_tr, average="macro", zero_division=0)
    prw_tr, rcw_tr, f1w_tr, _ = precision_recall_fscore_support(y_train, yhat_tr, average="weighted", zero_division=0)
    auc_tr = roc_auc_score(y_train, scores_tr, multi_class="ovr", average="macro")
    cm_tr = confusion_matrix(y_train, yhat_tr)

    # Test
    yhat_te = clf.predict(Xte)
    scores_te = clf.decision_function(Xte)
    r2s_te, r2_macro_te = clf.score_individual_r2(Xte, y_test)
    pr_te, rc_te, f1_te, _ = precision_recall_fscore_support(y_test, yhat_te, average="macro", zero_division=0)
    prw_te, rcw_te, f1w_te, _ = precision_recall_fscore_support(y_test, yhat_te, average="weighted", zero_division=0)
    auc_te = roc_auc_score(y_test, scores_te, multi_class="ovr", average="macro")
    cm_te = confusion_matrix(y_test, yhat_te)

    print(f"\n=== {name} ===")
    print("Train:  R2 macro =", round(r2_macro_tr, 4), "| F1 macro =", round(f1_tr, 4), "| Precision macro =", round(pr_tr,4), "| Recall macro =", round(rc_tr,4), "| ROC AUC macro =", round(auc_tr,4))
    print("Train (weighted): F1 =", round(f1w_tr,4), "Precision =", round(prw_tr,4), "Recall =", round(rcw_tr,4))
    print("Per-class R2 (train):", [round(v,4) for v in r2s_tr])
    print("Test:   R2 macro =", round(r2_macro_te, 4), "| F1 macro =", round(f1_te, 4), "| Precision macro =", round(pr_te,4), "| Recall macro =", round(rc_te,4), "| ROC AUC macro =", round(auc_te,4))
    print("Test (weighted):  F1 =", round(f1w_te,4), "Precision =", round(prw_te,4), "Recall =", round(rcw_te,4))
    print("Per-class R2 (test):", [round(v,4) for v in r2s_te])

    fig, ax = plt.subplots()
    ConfusionMatrixDisplay(cm_tr, display_labels=iris.target_names).plot(ax=ax, colorbar=False)
    ax.set_title(f"{name} — Train Confusion Matrix")
    plt.show()

    fig, ax = plt.subplots()
    ConfusionMatrixDisplay(cm_te, display_labels=iris.target_names).plot(ax=ax, colorbar=False)
    ax.set_title(f"{name} — Test Confusion Matrix")
    plt.show()

sc1, pc1, clf1, Xtr1, Xte1 = fit_pipeline(use_pca=False)
evaluate_pipeline("Baseline (Scaler → OVR Linear Regression)", sc1, pc1, clf1, Xtr1, Xte1)

sc2, pc2, clf2, Xtr2, Xte2 = fit_pipeline(use_pca=True, n_components=2)
evaluate_pipeline("With PCA (Scaler → PCA(2) → OVR Linear Regression)", sc2, pc2, clf2, Xtr2, Xte2)



## 8. Stratified K-Fold Cross-Validation
We compare **without** vs **with** PCA using 5-fold stratified CV.


In [ ]:

from sklearn.model_selection import StratifiedKFold

def cross_val_ovr_linearreg(X, y, use_pca=False, n_components=2, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    metrics = {"r2_macro": [], "f1_macro": [], "precision_macro": [], "recall_macro": [], "roc_auc_macro": []}
    for tr, te in skf.split(X, y):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]
        scaler = StandardScaler().fit(Xtr)
        Xtr_s = scaler.transform(Xtr)
        Xte_s = scaler.transform(Xte)
        if use_pca:
            pca = SKPCA(n_components=n_components, random_state=42).fit(Xtr_s)
            Xtr_s = pca.transform(Xtr_s)
            Xte_s = pca.transform(Xte_s)
        clf = OVRLinearRegression().fit(Xtr_s, ytr)
        yhat = clf.predict(Xte_s)
        scores = clf.decision_function(Xte_s)
        r2s, r2m = clf.score_individual_r2(Xte_s, yte)
        pr, rc, f1, _ = precision_recall_fscore_support(yte, yhat, average="macro", zero_division=0)
        auc = roc_auc_score(yte, scores, multi_class="ovr", average="macro")
        metrics["r2_macro"].append(r2m)
        metrics["f1_macro"].append(f1)
        metrics["precision_macro"].append(pr)
        metrics["recall_macro"].append(rc)
        metrics["roc_auc_macro"].append(auc)
    return {k: (float(np.mean(v)), float(np.std(v))) for k, v in metrics.items()}

cv_no_pca = cross_val_ovr_linearreg(X, y, use_pca=False, n_components=2, n_splits=5)
cv_pca    = cross_val_ovr_linearreg(X, y, use_pca=True,  n_components=2, n_splits=5)
print("CV (no PCA):", {k: (round(m,4), round(s,4)) for k,(m,s) in cv_no_pca.items()})
print("CV (with PCA):", {k: (round(m,4), round(s,4)) for k,(m,s) in cv_pca.items()})



## 9. Decision Regions in PCA(2)


In [ ]:

scaler_viz = StandardScaler().fit(X)
Xv = scaler_viz.transform(X)
pca_viz = SKPCA(n_components=2, random_state=42).fit(Xv)
Zv = pca_viz.transform(Xv)
clf_viz = OVRLinearRegression().fit(Zv, y)

x1min, x1max = Zv[:,0].min()-1, Zv[:,0].max()+1
x2min, x2max = Zv[:,1].min()-1, Zv[:,1].max()+1
xx1, xx2 = np.meshgrid(np.linspace(x1min, x1max, 200),
                       np.linspace(x2min, x2max, 200))
grid = np.c_[xx1.ravel(), xx2.ravel()]
pred = clf_viz.predict(grid).reshape(xx1.shape)

plt.contourf(xx1, xx2, pred, alpha=0.3)
plt.scatter(Zv[:,0], Zv[:,1], c=y, edgecolor="k")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("OVR Linear Regression in PCA(2) space (Iris)")
plt.show()



## 10. Why PCA Helps
- **Noise reduction** by discarding low-variance directions.
- **Better conditioning** for linear models.
- **Visualization** benefits in low dimensions.
- **Efficiency** in compute/storage.



## 11. Summary & Exercises
- PCA from scratch verified with sklearn.
- Fisher separability per PC connects variance (eigenvalues) to class separation.
- Baseline vs PCA(2) pipelines evaluated on train/test and via k-fold CV.

**Exercises**
1. Try `n_components=1` and `3` and re-evaluate.
2. Replace the OVR regressor with `LogisticRegression` and compare metrics.
3. Plot per-class ROC curves using the decision scores.
